# Phase 4 Feature Exploration — Final
## HFTExperiment v2 | 2026-05-13

All features use correct NPZ columns directly:
- DHPF: `vol_enc` + `gmm2` from NPZ (not broken vol_zscore)
- d_ATR: `high` + `low` raw prices from NPZ
- momentum: `close` raw prices from NPZ
- Silver proxy: `vol_enc`, `rq`, ATR-20 from raw prices
- Session: `timestamps_ns` from NPZ

## 0. Setup

In [1]:
import numpy as np
import matplotlib; matplotlib.use('Agg')
import matplotlib.pyplot as plt
from scipy.stats import ks_2samp, pearsonr
from sklearn.feature_selection import mutual_info_classif, mutual_info_regression
from pathlib import Path
import warnings; warnings.filterwarnings('ignore')

SEARCH = [
    r"D:\HFTExperiment\data\training_ready.npz",
    r"C:\HFTExperiment\data\training_ready.npz",
    r".\data\training_ready.npz",
    r"..\data\training_ready.npz",
    r".\training_ready.npz",
]
MANUAL_PATH = ""
NPZ_PATH = MANUAL_PATH or next((p for p in SEARCH if Path(p).exists()), None)
if not NPZ_PATH: raise FileNotFoundError("Set MANUAL_PATH to your training_ready.npz")
print(f"Loading: {NPZ_PATH}")
_d = np.load(NPZ_PATH, allow_pickle=True)
print("Keys:", list(_d.keys()))


Loading: D:\HFTExperiment\data\training_ready.npz
Keys: ['features', 'labels', 'close', 'high', 'low', 'timestamps_ns', 'gmm2', 'km_enc', 'vol_enc', 'gs_q', 'cu_au', 'rq', 'atr_norm', 'trend_norm', 'session_phase', 'tick_volume_raw', 'metadata']


## 1. Load all NPZ columns

In [2]:
FEAT_NAMES = ["open_sc","high_sc","low_sc","close_sc","vol_sc",
              "spread_sc","bar_return_bps","wick_asymmetry","vol_zscore","spread_pressure"]
labels = _d["labels"].astype(int)
fb     = _d["features"].astype(np.float64)
N      = min(len(labels), len(fb))
labels, fb = labels[:N], fb[:N]
sell_mask = labels==0; hold_mask = labels==1; buy_mask = labels==2
print(f"N={N:,}  sell={sell_mask.sum():,}({100*sell_mask.mean():.3f}%) hold={hold_mask.sum():,} buy={buy_mask.sum():,}")

# Load all extra NPZ columns
close_raw  = _d["close"].astype(np.float64)[:N]
high_raw   = _d["high"].astype(np.float64)[:N]
low_raw    = _d["low"].astype(np.float64)[:N]
ts_ns      = _d["timestamps_ns"].astype(np.int64)[:N]
gmm2       = _d["gmm2"].astype(np.float64)[:N]
vol_enc    = _d["vol_enc"].astype(np.float64)[:N]
tick_vol   = _d["tick_volume_raw"].astype(np.float64)[:N]
rq         = _d["rq"].astype(np.float64)[:N]
atr_n      = _d["atr_norm"].astype(np.float64)[:N]
trend_n    = _d["trend_norm"].astype(np.float64)[:N]
session_ph = _d["session_phase"].astype(np.float64)[:N]
print("All NPZ columns loaded.")
print(f"vol_enc:  min={vol_enc.min():.3f} max={vol_enc.max():.3f}")
print(f"gmm2:     min={gmm2.min():.3f} max={gmm2.max():.3f}")
print(f"tick_vol: min={tick_vol.min():.0f} max={tick_vol.max():.0f} mean={tick_vol.mean():.1f}")


N=5,680,771  sell=199,523(3.512%) hold=5,439,333 buy=41,915
All NPZ columns loaded.
vol_enc:  min=0.000 max=1.000
gmm2:     min=0.000 max=1.000
tick_vol: min=0 max=1748 mean=6.8


## 2. Evaluation helper

In [3]:
def eval_feature(name, values, labels, fb, thresh_mi=0.005, thresh_red=10.0):
    values = np.asarray(values,dtype=np.float64).ravel()
    n = min(len(values),len(labels),len(fb))
    v,l,fb_ = values[:n], np.asarray(labels[:n],int), fb[:n]
    ks_s,_=ks_2samp(v[l==0],v[l==1]); ks_b,_=ks_2samp(v[l==2],v[l==1])
    ks_max=max(ks_s,ks_b)
    # Use vol_enc for regime split (correct, not broken vol_zscore)
    ve = vol_enc[:n]; hv=ve>0.5
    ks_r=ks_2samp(v[hv],v[~hv])[0] if (hv.sum()>100 and (~hv).sum()>100) else 0.
    bl=(l==0).astype(np.int32)
    mi=mutual_info_classif(v.reshape(-1,1),bl,discrete_features=False,random_state=42)[0]
    nb=min(6,fb_.shape[1])
    mi_base=np.mean([mutual_info_classif(fb_[:,j:j+1].astype(float),bl,
             discrete_features=False,random_state=42)[0] for j in range(nb)])+1e-9
    mpair=1e-9; bj=0
    for j in range(fb_.shape[1]):
        mj=mutual_info_regression(fb_[:,j:j+1].astype(float),v,
                                  discrete_features=False,random_state=42)[0]
        if mj>mpair: mpair=mj; bj=j
    best=FEAT_NAMES[bj] if bj<len(FEAT_NAMES) else f"feat[{bj}]"
    ratio=mpair/(mi+1e-9); adj=max(0.,mi-mpair)
    verd="EXCLUDE"
    if ks_max>0.05 and mi>thresh_mi and ratio<thresh_red: verd="SUPERVISED"
    elif ks_max>0.05 and mi>thresh_mi: verd="RL OBS"
    elif ks_max>0.05: verd="EXCLUDE (MI fails)"
    sig="STRONG" if ks_max>0.10 else ("MOD" if ks_max>0.05 else "WEAK")
    print(f"\n{'='*60}\nFeature: {name}")
    print(f"  KS sell={ks_s:.4f} buy={ks_b:.4f} [{sig}] | regime={ks_r:.4f}")
    print(f"  MI={mi:.6f} ({mi/mi_base:.2f}x OHLCV) {'PASS' if mi>thresh_mi else 'FAIL'}")
    print(f"  MaxPairMI={mpair:.5f} vs {best} | ratio={ratio:.2f} {'PASS' if ratio<thresh_red else 'FAIL'}")
    print(f"  adj_MI={adj:.6f} | VERDICT: {verd}")
    return {"name":name,"ks_sell":ks_s,"ks_buy":ks_b,"ks_regime":ks_r,
            "mi":mi,"mi_vs_ohlcv":mi/mi_base,"ratio":ratio,"adj_mi":adj,
            "best_feat":best,"verdict":verd}
results={}; print("Helper ready. Using vol_enc for regime split.")


Helper ready. Using vol_enc for regime split.


## 3. GKV proxy
P6 Goel et al 2025.

In [4]:
# GKV proxy: 0.5*(H-L)^2 - (2*log2-1)*(C-O)^2 on MinMax-scaled OHLC
o=fb[:,0]; h=fb[:,1]; l=fb[:,2]; c=fb[:,3]
gkv_raw=0.5*(h-l)**2-(2*np.log(2)-1)*(c-o)**2
gkv_norm=np.tanh((gkv_raw-np.median(gkv_raw))/(np.std(gkv_raw)+1e-8))
print(f"GKV proxy: mean={gkv_norm.mean():.4f} std={gkv_norm.std():.4f}")
fig,axes=plt.subplots(1,2,figsize=(12,4))
for mask,nm,c_ in [(sell_mask,'sell','red'),(hold_mask,'hold','blue'),(buy_mask,'buy','green')]:
    axes[0].hist(gkv_norm[mask],bins=60,alpha=0.5,label=nm,color=c_)
axes[0].set_title("GKV proxy by label"); axes[0].legend()
axes[1].scatter(fb[:3000,6],gkv_norm[:3000],
    c=['red' if labels[i]==0 else 'blue' if labels[i]==1 else 'green' for i in range(3000)],
    alpha=0.3,s=4)
axes[1].set_xlabel("bar_return_bps"); axes[1].set_ylabel("GKV")
axes[1].set_title("GKV vs bar_return_bps")
plt.tight_layout(); plt.savefig("gkv_analysis.png",dpi=100); plt.show()
results["gkv_proxy"]=eval_feature("gkv_proxy",gkv_norm,labels,fb)


GKV proxy: mean=0.0843 std=0.3940

Feature: gkv_proxy
  KS sell=0.0739 buy=0.0795 [MOD] | regime=0.0046
  MI=0.000508 (0.05x OHLCV) FAIL
  MaxPairMI=0.21815 vs bar_return_bps | ratio=429.79 FAIL
  adj_MI=0.000000 | VERDICT: EXCLUDE (MI fails)


## 4. Directional ATR (rolling, raw prices)
P1 Ma et al 2021. Uses raw `high`/`low` columns.

In [5]:
# Rolling d_ATR: true 5-bar vs 15-bar ATR from raw high-low
W_R=5; W_B=15; WT=W_R+W_B
bar_range=(high_raw-low_raw).astype(np.float64)
atr_rec=np.zeros(N); atr_bas=np.zeros(N)
for i in range(WT,N):
    atr_rec[i]=bar_range[i-W_R+1:i+1].mean()
    atr_bas[i]=bar_range[i-WT:i-W_R+1].mean()
d_atr=np.tanh((atr_rec-atr_bas)/(np.abs(atr_bas)+1e-8)*5)
vm=np.zeros(N,dtype=bool); vm[WT:]=True
d_v=d_atr[vm]; lv=labels[vm]; fv=fb[vm]
sv=lv==0; hv_l=lv==1; bv=lv==2
print(f"Rolling d_ATR (raw prices): mean={d_v.mean():.4f} std={d_v.std():.4f}")

ve_v=vol_enc[vm]; g_v=gmm2[vm]; ret_v=fb[vm,6]
bear_v=g_v<0.5; high_v=ve_v>0.75; ris_v=d_v>0
bh=bear_v&high_v; bhr=(bh&ris_v).sum(); bhf=(bh&~ris_v).sum()
fig,axes=plt.subplots(1,2,figsize=(12,4))
for mask,nm,c_ in [(sv,'sell','red'),(hv_l,'hold','blue'),(bv,'buy','green')]:
    axes[0].hist(d_v[mask],bins=60,alpha=0.5,label=nm,color=c_)
axes[0].axvline(x=0,color='k',linestyle='--',alpha=0.5)
axes[0].set_title("Rolling d_ATR (raw prices) by label"); axes[0].legend()
axes[1].bar(['Rising\n(block)','Falling\n(allow)'],[bhr,bhf],color=['red','green'],alpha=0.85)
for i,val in enumerate([bhr,bhf]):
    if bh.sum()>0: axes[1].text(i,val*0.5,f"{val:,}\n({100*val/bh.sum():.0f}%)",
                                ha='center',va='center',fontweight='bold',color='white')
axes[1].set_title(f"Bear+HIGH gate (raw ATR)\nTotal={bh.sum():,}")
plt.tight_layout(); plt.savefig("d_atr_analysis.png",dpi=100); plt.show()
print(f"Rising={bhr:,} Falling={bhf:,} / Bear+HIGH={bh.sum():,}")
results["d_atr_norm"]=eval_feature("d_atr_norm",d_v,lv,fv)


Rolling d_ATR (raw prices): mean=-0.0607 std=0.7626
Rising=324,380 Falling=399,948 / Bear+HIGH=724,328

Feature: d_atr_norm
  KS sell=0.0528 buy=0.0528 [MOD] | regime=0.0040
  MI=0.008624 (0.93x OHLCV) PASS
  MaxPairMI=0.21242 vs spread_pressure | ratio=24.63 FAIL
  adj_MI=0.000000 | VERDICT: RL OBS


## 5. 5-bar Momentum (raw close returns)
P7 Zhao et al 2025. Uses raw `close` column.

In [6]:
# 5-bar momentum from raw close prices (true returns)
W_M=5
log_ret=np.concatenate([[0.],np.log(close_raw[1:]/(close_raw[:-1]+1e-8))])
mom5=np.zeros(N)
for i in range(W_M,N): mom5[i]=log_ret[i-W_M+1:i+1].sum()*10000  # bps
mom_std=np.std(mom5[W_M:])+1e-8; mom5_n=np.tanh(mom5/mom_std)
vm2=np.zeros(N,dtype=bool); vm2[W_M:]=True
mv=mom5_n[vm2]; lm=labels[vm2]; fm=fb[vm2]
print(f"5-bar momentum (raw returns): mean={mv.mean():.4f} std={mv.std():.4f}")
fig,ax=plt.subplots(figsize=(8,4))
for mask,nm,c_ in [((lm==0),'sell','red'),((lm==1),'hold','blue'),((lm==2),'buy','green')]:
    ax.hist(mv[mask],bins=60,alpha=0.5,label=nm,color=c_)
ax.set_title("5-bar momentum (raw log-returns) by label"); ax.legend()
plt.tight_layout(); plt.savefig("momentum_analysis.png",dpi=100); plt.show()
results["momentum_5bar"]=eval_feature("momentum_5bar",mv,lm,fm)


5-bar momentum (raw returns): mean=0.0032 std=0.4942

Feature: momentum_5bar
  KS sell=0.1669 buy=0.1954 [STRONG] | regime=0.0860
  MI=0.011947 (1.29x OHLCV) PASS
  MaxPairMI=0.91765 vs bar_return_bps | ratio=76.81 FAIL
  adj_MI=0.000000 | VERDICT: RL OBS


## 6. DHPF 4-Partition — FIXED
Uses `vol_enc` and `gmm2` directly from NPZ. Resolves all previous artifacts.

In [7]:
# DHPF 4-Partition -- FIXED: uses vol_enc and gmm2 directly from NPZ
# vol_enc: 0=LOW, 0.5=NORMAL, 1=HIGH (pre-encoded, correct)
# gmm2:    0=Bear, 1=Bull

bull   = gmm2 > 0.5
low_v  = vol_enc < 0.25
norm_v = (vol_enc >= 0.25) & (vol_enc < 0.75)
high_v = (vol_enc >= 0.75) & (vol_enc < 0.95)
shock_v = vol_enc >= 0.95   # top 5% of HIGH vol = event shock

parts = {
    "Bull-LOW":    bull  & low_v,
    "Bull-NORMAL": bull  & norm_v,
    "Bull-HIGH":   bull  & high_v,
    "Bull-SHOCK":  bull  & shock_v,
    "Bear-LOW":   ~bull  & low_v,
    "Bear-NORMAL":~bull  & norm_v,
    "Bear-HIGH":  ~bull  & high_v,
    "Bear-SHOCK": ~bull  & shock_v,
}

base_sell = (labels==0).mean()
print(f"Baseline sell: {100*base_sell:.3f}%\n")
print(f"{'Partition':<15}{'N':>9}{'%tot':>7}{'Sell%':>8}{'Hold%':>8}{'Rec.mult':>10}")
print("-"*60)
for pn,pm in parts.items():
    n_=pm.sum()
    if n_==0: continue
    sr=(labels[pm]==0).mean(); hr=(labels[pm]==1).mean()
    mult=float(np.clip(base_sell*2/(sr+1e-8),1.,8.))
    tag=" <-- SHOCK" if "SHOCK" in pn else ""
    print(f"  {pn:<13}{n_:>9,}{100*n_/N:>6.1f}%{100*sr:>7.3f}%{100*hr:>7.3f}%{mult:>10.2f}x{tag}")

pnames=list(parts.keys())
srates=[100*(labels[m]==0).mean() if m.sum()>0 else 0. for m in parts.values()]
sizes=[m.sum() for m in parts.values()]
fig,axes=plt.subplots(1,2,figsize=(15,4))
axes[0].bar(pnames,srates,color=['#2ecc71' if 'Bull' in p else '#e74c3c' for p in pnames],alpha=0.85)
axes[0].axhline(y=100*base_sell,color='k',linestyle='--',linewidth=2,label=f"Baseline={100*base_sell:.3f}%")
axes[0].set_ylabel("Sell label rate (%)"); axes[0].legend()
axes[0].set_title("DHPF: Sell label by partition (vol_enc + gmm2)")
plt.setp(axes[0].get_xticklabels(),rotation=35,ha='right')
axes[1].bar(pnames,sizes,color=['#2ecc71' if 'Bull' in p else '#e74c3c' for p in pnames],alpha=0.85)
axes[1].set_ylabel("N bars"); axes[1].set_title("Partition sizes")
plt.setp(axes[1].get_xticklabels(),rotation=35,ha='right')
plt.tight_layout(); plt.savefig("dhpf_partitions.png",dpi=100); plt.show()

shock_mask = vol_enc >= 0.95
print(f"\nShock (vol_enc >= 0.95): {shock_mask.sum():,} ({100*shock_mask.mean():.2f}%)")
print(f"Sell in shock: {(labels[shock_mask]==0).sum():,} ({100*(labels[shock_mask]==0).mean():.3f}%)")
print(f"Shock/baseline sell ratio: {(labels[shock_mask]==0).mean()/base_sell:.2f}x")
print("\nRecommended sampler multipliers:")
for pn,pm in parts.items():
    if pm.sum()==0: continue
    sr=(labels[pm]==0).mean()
    mult=float(np.clip(base_sell*2/(sr+1e-8),1.,8.))
    if mult > 1.5: print(f"  {pn}: x{mult:.1f}")


Baseline sell: 3.512%

Partition              N   %tot   Sell%   Hold%  Rec.mult
------------------------------------------------------------
  Bull-LOW     2,068,405  36.4%  1.042% 98.839%      6.74x
  Bull-NORMAL  2,026,350  35.7%  2.280% 97.473%      3.08x
  Bull-SHOCK     848,262  14.9%  3.207% 96.355%      2.19x <-- SHOCK
  Bear-NORMAL     13,406   0.2%  1.537% 97.971%      4.57x
  Bear-SHOCK     724,348  12.8% 14.406% 81.361%      1.00x <-- SHOCK

Shock (vol_enc >= 0.95): 1,572,610 (27.68%)
Sell in shock: 131,560 (8.366%)
Shock/baseline sell ratio: 2.38x

Recommended sampler multipliers:
  Bull-LOW: x6.7
  Bull-NORMAL: x3.1
  Bull-SHOCK: x2.2
  Bear-NORMAL: x4.6


## 7. Silver Proxy MI Test
P7: silver futures rank 3 of 36 in XGBoost importance.
Tests proxies available in current NPZ before committing to XAGUSD feed rebuild.

In [8]:
# Silver Proxy MI Test -- P7 Zhao et al 2025
# True XAGUSD requires a separate M1 feed + NPZ rebuild.
# Proxy: gold-silver ratio proxy from volatility structure.
# Gold and silver are highly correlated (r~0.85-0.95 rolling).
# We test whether intraday vol_enc captures cross-metal signal already.
# Also test: ATR normalised by 20-bar baseline (captures vol regime)

print("=== Silver Proxy: testing closest available approximation ===\n")

# Proxy 1: vol_enc * bar_return_bps (vol-weighted directional return)
# Approximates what silver futures add: a cross-metal momentum signal
silver_p1 = np.tanh(vol_enc * fb[:,6] / (np.std(fb[:,6])+1e-8))
print(f"vol_enc x ret_bps: mean={silver_p1.mean():.4f} std={silver_p1.std():.4f}")
results["vol_enc_x_ret"] = eval_feature("vol_enc_x_ret", silver_p1, labels, fb)

# Proxy 2: Normalised ATR from raw prices (captures same regime as XAG correlation)
# High ATR in gold = high ATR in silver = high cross-metal correlation
W_ATR=20
bar_r=(high_raw-low_raw).astype(np.float64)
atr20=np.array([bar_r[max(0,i-W_ATR+1):i+1].mean() for i in range(N)])
atr20_norm=np.tanh((atr20-np.median(atr20))/(np.std(atr20)+1e-8))
print(f"\nNorm ATR-20 (silver regime proxy): mean={atr20_norm.mean():.4f} std={atr20_norm.std():.4f}")
results["atr20_norm"] = eval_feature("atr20_norm", atr20_norm, labels, fb)

# Proxy 3: rq (regime quality) from NPZ -- already computed, tests if it adds info
print(f"\nrq (regime quality from NPZ): mean={rq.mean():.4f} std={rq.std():.4f}")
results["rq_regime"] = eval_feature("rq_regime", rq, labels, fb)

print("\nNote: True XAGUSD test requires separate M1 feed.")
print("If any proxy SUPERVISED: silver futures is a high-priority addition.")
print("If all EXCLUDE/RL OBS: existing features already capture cross-metal signal.")


=== Silver Proxy: testing closest available approximation ===

vol_enc x ret_bps: mean=0.0009 std=0.3522

Feature: vol_enc_x_ret
  KS sell=0.1945 buy=0.2416 [STRONG] | regime=0.2688
  MI=0.016624 (1.79x OHLCV) PASS
  MaxPairMI=7.56122 vs bar_return_bps | ratio=454.84 FAIL
  adj_MI=0.000000 | VERDICT: RL OBS

Norm ATR-20 (silver regime proxy): mean=0.1090 std=0.3737

Feature: atr20_norm
  KS sell=0.6534 buy=0.7641 [STRONG] | regime=0.2908
  MI=0.049270 (5.30x OHLCV) PASS
  MaxPairMI=0.62163 vs spread_pressure | ratio=12.62 FAIL
  adj_MI=0.000000 | VERDICT: RL OBS

rq (regime quality from NPZ): mean=0.3188 std=0.1708

Feature: rq_regime
  KS sell=0.4147 buy=0.6231 [STRONG] | regime=1.0000
  MI=0.197832 (21.29x OHLCV) PASS
  MaxPairMI=0.11534 vs low_sc | ratio=0.58 PASS
  adj_MI=0.082491 | VERDICT: SUPERVISED

Note: True XAGUSD test requires separate M1 feed.
If any proxy SUPERVISED: silver futures is a high-priority addition.
If all EXCLUDE/RL OBS: existing features already capture cross

## 8. Session-Aware Analysis
P1: asymmetric uncertainty across sessions.
Uses `timestamps_ns` to extract UTC hour. Validates VIO session gate and sell label concentration.

In [9]:
# Session-Aware Sell Label Concentration -- P1 Ma et al 2021
# Using timestamps_ns from NPZ to extract UTC hour.
# Tests: do sell labels concentrate in specific trading sessions?
# Bangkok UTC+7: Asian=00:00-08:00 UTC (07:00-15:00 BKK)
#                London=08:00-16:00 UTC (15:00-23:00 BKK)
#                NY=13:00-22:00 UTC    (20:00-05:00 BKK)

utc_hours = ((ts_ns // (3600 * 10**9)) % 24).astype(np.int32)
print(f"UTC hours: min={utc_hours.min()} max={utc_hours.max()}")
print(f"Unique hours: {sorted(set(utc_hours))}")

sessions = {
    "Asian (00-08 UTC)":   (utc_hours >= 0)  & (utc_hours < 8),
    "London (08-16 UTC)":  (utc_hours >= 8)  & (utc_hours < 16),
    "NY (16-22 UTC)":      (utc_hours >= 16) & (utc_hours < 22),
    "London+NY overlap\n(13-16 UTC)": (utc_hours >= 13) & (utc_hours < 16),
}

base_sell = (labels==0).mean()
print(f"\nBaseline sell: {100*base_sell:.3f}%\n")
print(f"{'Session':<25}{'N':>9}{'%tot':>7}{'Sell%':>8}{'Hold%':>8}{'vs base':>8}")
print("-"*65)
for sn,sm in sessions.items():
    n_=sm.sum()
    if n_==0: continue
    sr=(labels[sm]==0).mean(); hr=(labels[sm]==1).mean()
    ratio_=sr/base_sell
    sn_short=sn.split("\n")[0]
    print(f"  {sn_short:<23}{n_:>9,}{100*n_/N:>6.1f}%{100*sr:>7.3f}%{100*hr:>7.3f}%{ratio_:>8.2f}x")

# Plot hourly sell rate
hourly_sell = []
hours_present = sorted(set(utc_hours))
for h in hours_present:
    mask=utc_hours==h
    hourly_sell.append(100*(labels[mask]==0).mean() if mask.sum()>100 else 0.)

fig,axes=plt.subplots(1,2,figsize=(15,4))
axes[0].bar(hours_present,hourly_sell,color=['#e74c3c' if s>100*base_sell*1.1 else '#3498db' for s in hourly_sell])
axes[0].axhline(y=100*base_sell,color='k',linestyle='--',linewidth=2,label=f"Baseline={100*base_sell:.3f}%")
axes[0].set_xlabel("UTC Hour"); axes[0].set_ylabel("Sell label rate (%)")
axes[0].set_title("Sell label concentration by UTC hour")
axes[0].legend(); axes[0].set_xticks(hours_present)

# Vol_enc by session -- shows why Asian session has low signal
hourly_vol = [vol_enc[utc_hours==h].mean() if (utc_hours==h).sum()>100 else 0. for h in hours_present]
axes[1].bar(hours_present,hourly_vol,color='#2ecc71',alpha=0.8)
axes[1].set_xlabel("UTC Hour"); axes[1].set_ylabel("Mean vol_enc")
axes[1].set_title("Volatility by UTC hour (confirms Asian session dead zone)")
axes[1].set_xticks(hours_present)
plt.tight_layout(); plt.savefig("session_analysis.png",dpi=100); plt.show()

print("\nVIO session-aware gate implication:")
asian_mask = (utc_hours >= 0) & (utc_hours < 8)
active_mask = ~asian_mask
print(f"  Asian session bars (00-08 UTC): {asian_mask.sum():,} ({100*asian_mask.mean():.1f}%)")
print(f"  Active session bars (08-22 UTC): {active_mask.sum():,} ({100*active_mask.mean():.1f}%)")
print(f"  Asian sell rate: {100*(labels[asian_mask]==0).mean():.3f}%")
print(f"  Active sell rate: {100*(labels[active_mask]==0).mean():.3f}%")
print(f"  Sell concentration in active: {(labels[active_mask]==0).mean()/base_sell:.2f}x baseline")

# Eval session_phase feature already in NPZ
print(f"\nNPZ session_phase feature: mean={session_ph.mean():.4f} std={session_ph.std():.4f}")
results["session_phase_npz"] = eval_feature("session_phase_npz", session_ph, labels, fb)


UTC hours: min=0 max=23
Unique hours: [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23]

Baseline sell: 3.512%

Session                          N   %tot   Sell%   Hold% vs base
-----------------------------------------------------------------
  Asian (00-08 UTC)      2,098,422  36.9%  2.692% 96.744%    0.77x
  London (08-16 UTC)     2,066,013  36.4%  5.070% 94.040%    1.44x
  NY (16-22 UTC)         1,096,684  19.3%  2.658% 96.573%    0.76x
  London+NY overlap        761,580  13.4%  2.656% 96.795%    0.76x

VIO session-aware gate implication:
  Asian session bars (00-08 UTC): 2,098,422 (36.9%)
  Active session bars (08-22 UTC): 3,582,349 (63.1%)
  Asian sell rate: 2.692%
  Active sell rate: 3.993%
  Sell concentration in active: 1.14x baseline

NPZ session_phase feature: mean=0.3493 std=0.3749

Feature: session_phase_npz
  KS sell=0.1172 buy=0.0896 [STRONG] | regime=0.0066
  MI=0.037076 (3.99x OHLCV) PASS
  MaxPairMI=0.10980 vs spread_pressure | rat

## 9. Final Summary + Actions

In [10]:
print("="*65)
print("PHASE 4 FINAL VERDICTS (all features, corrected NPZ)")
print("="*65)
print(f"{'Feature':<25}{'KS':>8}{'MI':>10}{'Ratio':>8}  Verdict")
print("-"*65)
for name,r in results.items():
    ks=max(r['ks_sell'],r['ks_buy'])
    print(f"  {name:<23}{ks:>8.4f}{r['mi']:>10.6f}{r['ratio']:>8.1f}  {r['verdict']}")

print()
print("="*65)
print("ACTIONS")
print("="*65)
supervised_adds = [name for name,r in results.items() if r['verdict']=='SUPERVISED']
rl_adds         = [name for name,r in results.items() if r['verdict']=='RL OBS']
excluded        = [name for name,r in results.items() if 'EXCLUDE' in r['verdict']]

print(f"SUPERVISED additions: {supervised_adds if supervised_adds else 'none'}")
print(f"RL OBS additions:     {rl_adds if rl_adds else 'none'}")
print(f"Excluded:             {excluded}")

print()
print("DHPF sampler: check partition table above for Bear-SHOCK multiplier")
print("  If Bear-SHOCK sell% > baseline: add x4.0 to train_supervised.py")
print("  If Bear-SHOCK sell% <= baseline: skip DHPF addition")
print()
print("Session VIO: check session table above")
print("  If active session sell% > baseline: VIO enabled on London+NY only is justified")
print()
print("RUN 11 command (fresh, no resume):")
print("  python scripts/train_supervised.py model=dual_branch data=xauusd")
print("         training.batch_size=4096 training.epochs=200")
print()
print("RL PHASE 9 (after Run 11, use _best.pt):")
print("  python scripts/train_rl.py --checkpoint .../dual_branch_best.pt")
print("         --steps 1500000 --n-evolves 10 --commission 0.70 --spread-pips 2.0")


PHASE 4 FINAL VERDICTS (all features, corrected NPZ)
Feature                        KS        MI   Ratio  Verdict
-----------------------------------------------------------------
  gkv_proxy                0.0795  0.000508   429.8  EXCLUDE (MI fails)
  d_atr_norm               0.0528  0.008624    24.6  RL OBS
  momentum_5bar            0.1954  0.011947    76.8  RL OBS
  vol_enc_x_ret            0.2416  0.016624   454.8  RL OBS
  atr20_norm               0.7641  0.049270    12.6  RL OBS
  rq_regime                0.6231  0.197832     0.6  SUPERVISED
  session_phase_npz        0.1172  0.037076     3.0  SUPERVISED

ACTIONS
SUPERVISED additions: ['rq_regime', 'session_phase_npz']
RL OBS additions:     ['d_atr_norm', 'momentum_5bar', 'vol_enc_x_ret', 'atr20_norm']
Excluded:             ['gkv_proxy']

DHPF sampler: check partition table above for Bear-SHOCK multiplier
  If Bear-SHOCK sell% > baseline: add x4.0 to train_supervised.py
  If Bear-SHOCK sell% <= baseline: skip DHPF addition

Ses